# NLP-03 · LLM Pretraining and Data Pipeline
### From a versioned corpus to a domain-adapted checkpoint

The tiny-language-model gate proved next-token training. This lesson asks what happens
before and after that loop: curate a corpus, prevent evaluation contamination, estimate
training work, continue a real checkpoint on domain text, and measure both improvement
and forgetting. **Prerequisites:** NLP-06, FND-04, MLE-02, and PROD-04.


> **Canonical learner route · module NLP-03 of 67**
>
> Required prerequisites: **FND-04, MLE-02, PROD-04, DL-08, NLP-01, NLP-06** · Previous: **NLP-02** ·
> Next after mastery: **NLP-07** · Expected first-pass workload:
> **5–8 hours**
>
> **Core path:** intuition, objectives, foundations, runnable implementation,
> failure modes, and assessed exercises. **Extension path:** history, production,
> tradeoffs, and interview material may be revisited after the core pass.
> Do not continue merely because every cell ran. Continue when you can complete
> the independent exercise and teach-back without notes. The canonical route in
> `docs/CURRICULUM_PATH.json` is authoritative when section-local file order and
> prerequisite order differ.


## 1 · Learning Objectives

- Distinguish base pretraining from continued pretraining.
- Apply explicit provenance, quality, deduplication, and contamination rules.
- Keep tokenizer/checkpoint/data versions compatible.
- calculate next-token loss and a first-order compute budget.
- Measure domain improvement and original-domain retention separately.
- Choose continued pretraining only when data and evidence justify it.


<details>
<summary><strong>Mathematics notation support — open when a formula feels dense</strong></summary>

- $x_i$: item or component number $i$; a subscript is a label, not multiplication.
- $n$: number of examples; $d$: number of features or dimensions.
- $\mathbf x$: vector; $X$: matrix; $X^\top$: transpose (rows and columns swap).
- $\hat y$: an estimate or prediction; a hat marks an estimated quantity.
- $\sum$: add repeated terms; $\prod$: multiply repeated terms.
- $\lVert\mathbf x\rVert$: vector length; $|x|$: distance of one number from zero.
- $\frac{\partial f}{\partial x}$: slope of $f$ as $x$ changes while other inputs
  stay fixed; $\nabla f$: vector containing all parameter slopes.
- $P(A\mid B)$: probability of $A$ after restricting attention to cases where
  $B$ is true.
- $\log$ reverses an exponential and turns products into sums.

Read a formula one operator at a time, write object shapes beside vectors and
matrices, and substitute a tiny numeric example. Review PRE-01 through PRE-04 for
worked explanations of these symbols.
</details>


## Student Lesson Companion · NLP-03 · LLM Pretraining and Data Pipeline

Use this companion during the **core pass**. Write short answers before reading
the extension material; then correct them in a different colour after the lesson.

### Practical problem before history

1. What concrete task or decision are we trying to improve?
2. Why is it difficult with the data, time, labels, or compute available?
3. What is the simplest previous baseline?
4. Where does that baseline fail?
5. What capability must the new concept add?

Section 2 must supply enough evidence to answer these questions. Historical detail
is extension material unless it explains the problem or design constraint.

### Concept, analogy, and analogy limit

After Section 3, explain the concept in one sentence without unexplained jargon.
Map it to one daily-life analogy **or** one concrete visual example. Then state
one place where the mapping breaks; an analogy is a bridge, not a proof.

### Use / avoid / alternatives

Complete this decision table from Sections 7, 9, and 11:

| Decision | Required answer |
|---|---|
| Use it when | Three realistic conditions where its assumptions and benefits fit |
| Avoid it when | Two conditions where it is misleading, unsafe, or wasteful |
| Prefer instead | At least one simpler baseline and one alternative for a failed assumption |
| Evidence | Metric, diagnostic, or constraint that supports the choice |


## 2 · Historical Motivation

Decoder pretraining made one objective reusable across tasks; scaling-law research then
showed that parameters, tokens, and compute must be budgeted together. Domain adaptation
reuses that checkpoint instead of restarting. The practical lesson is not “more text is
always better”: duplicated, unlicensed, poisoned, or evaluation-contaminated data can
make a larger run less trustworthy.


## 3 · Intuition and Practical Problem

A general model reads ordinary prose but performs poorly on astronomy language. We can
continue next-token training on licensed astronomy text. This is like a trained musician
studying a new genre: prior skill helps, but exclusive practice can weaken the old
repertoire. The analogy stops at mechanism—the model only changes parameters to reduce
loss.

Use continued pretraining for domain language, style, or vocabulary at scale. Avoid it
when facts change often (prefer RAG), when a prompt solves the task, or when you cannot
build domain and retention evaluation sets.


## 4 · Mathematical Foundations

After reading tokens $x_1,ldots,x_t$, the model predicts $x_{t+1}$. Read the loss aloud:
“average the negative log probability assigned to every true next token.”

$$J=-\frac1{T-1}\sum_{t=1}^{T-1}\log p_\theta(x_{t+1}\mid x_{1:t}).$$

$T$ is sequence length; $t$ is position; $x_{1:t}$ is visible context; $p_\theta$ is
the probability from parameters $\theta$; $J$ is mean loss in nats. If true-token
probabilities are `0.5` and `0.25`, $J=(-\log .5-\log .25)/2=1.040$ nats.

A rough dense-Transformer training estimate is $C\approx6ND$ FLOPs: $N$ parameters,
$D$ training tokens, and factor 6 approximating forward plus backward work. For 1M
parameters and 10M tokens, $C\approx6\times10^{13}$ FLOPs. It is planning arithmetic,
not a hardware-runtime guarantee; architecture and utilization matter.


## 5 · Manual Implementation from Scratch

The local lab normalizes documents, rejects a short item, removes an exact duplicate,
and blocks an exact evaluation substring. It then trains the existing tiny decoder on
base text, copies its checkpoint, and continues training only the copy on curated domain
text.


In [ ]:
import sys
from pathlib import Path
roots=[Path.cwd(), *Path.cwd().parents]
repo=next(p for p in roots if (p/'projects/language_model_adaptation').exists())
sys.path[:0]=[str(repo/'projects/language_model_adaptation/src'), str(repo/'projects/tiny_language_model/src')]
from language_model_adaptation.lab import run_adaptation_lab
report=run_adaptation_lab(seed=42)
print(report['data_pipeline'])
print(report['continued_pretraining'])


## 6 · Visualization

The domain bar should fall; the retention bar may rise. Both are evidence—the second is
not an inconvenient metric to hide.


In [ ]:
import matplotlib.pyplot as plt
m=report['continued_pretraining']
labels=['domain','base retention']
before=[m['domain_loss_before'],m['base_retention_loss_before']]
after=[m['domain_loss_after'],m['base_retention_loss_after']]
x=range(2)
plt.bar([i-.18 for i in x],before,.36,label='before')
plt.bar([i+.18 for i in x],after,.36,label='after')
plt.xticks(list(x),labels); plt.ylabel('held-out token loss'); plt.legend(); plt.show()


## 7 · Failure Modes and Common Mistakes

| Symptom | Cause | Evidence | Repair |
|---|---|---|---|
| suspicious validation gain | contamination | overlap report | rebuild split before training |
| domain improves, base worsens | forgetting | retention loss | mix replay data or reduce updates |
| checkpoint will not load | tokenizer/config mismatch | version manifest | restore exact paired artifacts |
| loss falls only on duplicates | memorization | duplicate clusters | deduplicate before split |

Beginners often split after windowing, fit a new tokenizer that changes IDs, select on
the final test set, or report domain loss without retention.


## 8 · Library or Tool Implementation

Production pipelines may use dataset manifests, MinHash, distributed trainers, bf16,
FSDP, and checkpoint sharding. Those tools scale the same contract; they do not replace
provenance, split integrity, or a local baseline. Pin revisions and keep the core lesson
offline.


## 9 · Realistic Case Study

A legal team considers adaptation on licensed contracts. It first blocks benchmark and
client-test overlap, compares exact and near-duplicate clusters, holds out contract types,
and measures general-language retention. If the need is fresh case retrieval rather than
language adaptation, it chooses RAG instead.


## 10 · Learning and Production Considerations

Record corpus hashes, licenses, filters, mixture weights, tokenizer, model config,
optimizer, precision, seeds, tokens, FLOPs, and checkpoints. Exact overlap detection is
only a floor; paraphrase and benchmark contamination need stronger audits.


## 11 · Tradeoff Analysis

| Method | Solves | Strength | Risk |
|---|---|---|---|
| prompt | task framing | cheapest | limited behavior change |
| RAG | changing knowledge | source updates | retrieval dependency |
| continued pretraining | domain language | broad domain exposure | forgetting and compute |
| train from scratch | new base model | full control | enormous data/compute need |


## 12 · Readiness Check

Explain why the experiment's domain loss `4.043→1.758` is good evidence and its base
loss `1.853→3.189` is a warning. A strong answer proposes a replay-mixture experiment,
not deletion of the retention metric.


## 13 · Teach-Back

1. Why curate before splitting and training?
2. What does exact deduplication miss?
3. Why keep the tokenizer fixed?
4. What does $6ND$ estimate and omit?
5. When should RAG replace continued pretraining?


## 14 · Exercises, Self-Check, and Solutions

**Worked (10 min):** calculate $6ND$ for `N=2M`, `D=5M`: `6×10^13` FLOPs.
**Guided (20 min):** add one duplicate; hint: normalize before hashing. Expected:
duplicate count rises by one. **Independent (45 min):** mix 20% base replay and report
both losses. Pass if domain still improves and retention damage shrinks. **Challenge
(60 min):** add paraphrase-contamination detection and document false positives.

Summary: continued pretraining adapts language distribution, but every domain gain must
be paired with split integrity and retention evidence. **Memory aid:** *Curate, version,
continue, then measure both the new skill and the old one.*


## Lesson Close · Summary, Student Check, and Memory Aid

### Five short student checks

1. What practical problem does **NLP-03 · LLM Pretraining and Data Pipeline** solve?
2. What is its central mechanism in simple language?
3. Which assumption or limitation is easiest to forget?
4. What output or diagnostic tells you it worked as intended?
5. When would you choose a simpler or related alternative?

### Plain-language summary

Complete four sentences without notes: **The problem is… The concept works by…
I would use it when… I would avoid it when…** Compare your answer with the
objectives, failure modes, tradeoff analysis, and teach-back section.

### One-sentence memory aid

**Remember NLP-03 · LLM Pretraining and Data Pipeline: start from the problem, trace the mechanism, verify the
evidence, and use it only when its assumptions fit.** Replace this general aid
with your own topic-specific sentence of no more than 20 words.

The lesson is complete only after the Required Core Mastery Gate, not after the
final code cell runs.


## Required Core Mastery Gate · NLP-03

Complete this gate before treating the module as finished. The longer exercises
in Section 14 are extension practice unless the module says otherwise.

**Worked example:** rerun the smallest worked example in this notebook. Annotate
every input, output, shape or unit, and the claim the result supports.

**Guided practice (20–30 min):** change one input in that example. Before running
it, predict the direction of change and explain why. Use the nearest preceding
formula or algorithm step as a hint. **Self-check:** compare prediction with the
result and explain any mismatch rather than editing the prediction afterward.

**Independent practice (30–45 min):** on fresh tiny data, reproduce the module's
central operation without copying the completed cell. State assumptions, expected
output, and one invalid input. **Self-check:** verify with an assertion, a manual
calculation, or a trusted library implementation.

**Challenge (30–60 min):** create one failure described in Section 7, detect it
using evidence, and repair it without changing unrelated variables.

**Solution and scoring rubric:** 2 points for a correct prediction, 2 for a
runnable independent implementation, 2 for an objective self-check, 2 for failure
diagnosis, and 2 for teach-back without notes. Common mistakes: copying before
attempting, using later-module concepts as unexplained shortcuts, evaluating on
training data, and continuing because cells ran. **Readiness threshold: 8/10**,
including both independent implementation and teach-back points. If below 8,
revisit the named prerequisite in the route card and retry with different data.
